# From next-token prediction to following instructions

The models of the previous notebook (`next-token-generation.ipynb`) were
trained for one task only: give the probability of the **next** token of a
text from the web. Ask them a question and they do not *answer* it: they
**continue** the text, as if the question were the beginning of a web page.

Assistants such as ChatGPT are the same kind of decoder, with a second
training step: **instruction fine-tuning** on examples of instructions and
good responses (OpenAI, [*Aligning language models to follow
instructions*](https://openai.com/index/instruction-following/), the
InstructGPT models, 2022):

```
1. pre-training         a lot of text (web pages)                 ──► BASE model: continues texts
                        task: predict the next token
2. instruction tuning   (instruction, response) examples          ──► INSTRUCT model: answers
                        task: the SAME — predict the next token,
                        but only on the response
```

InstructGPT has three steps after pre-training: (1) **supervised
fine-tuning** (SFT) on responses written by people; (2) a *reward model*
trained on people's rankings of several responses; (3) reinforcement
learning against that reward model (RLHF). This notebook compares tiny base
models with their instruction-tuned versions:

| part | base model (step 1 only) | instruction-tuned version | how it was tuned |
|---|---|---|---|
| 2–3 | `distilgpt2` (82M), or `gpt2` (124M) | `MBZUAI/LaMini-GPT-124M` (124M) | `gpt2` + SFT on 2.58M instructions whose responses were written by `gpt-3.5-turbo` |
| 4 | `HuggingFaceTB/SmolLM2-135M` | `HuggingFaceTB/SmolLM2-135M-Instruct` | SFT, then DPO with people's preferences (a simpler replacement for steps 2–3) |

`LaMini-GPT-124M` **is** `gpt2`: same 12 layers, same vocabulary (50,257
tokens, plus one padding token), same 124M parameters — only the weights
changed during the fine-tuning. `distilgpt2` is the distilled `gpt2` of the previous notebook.

## 1. Preparation

Only one model is kept in memory at a time: `load` replaces the previous
one. Text is generated with **greedy decoding** (τ = 0 in the previous
notebook): every run gives the same output.

`complete` returns the continuation and says whether the model **stopped by
itself** (it chose its end-of-text token) or was cut at the length limit
(`MAX_NEW_TOKENS`).

In [1]:
import gc
import textwrap
import warnings
warnings.filterwarnings("ignore")

import huggingface_hub
import pandas as pd
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

transformers.logging.set_verbosity_error()
transformers.logging.disable_progress_bar()
huggingface_hub.logging.set_verbosity_error()
pd.set_option("display.max_colwidth", 100)         # show the answers in full in the tables

MAX_NEW_TOKENS = 60
model, tokenizer = None, None

def load(name):
    # Loads a model and its tokenizer, replacing the previous model in memory.
    global model, tokenizer
    model = None
    gc.collect()
    tokenizer = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name).eval()   # eval(): inference only
    print(f"{name}: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters, "
          f"{model.config.num_hidden_layers} layers, vocabulary of {len(tokenizer):,} tokens")

def end_of_text_ids():
    eos = model.generation_config.eos_token_id
    return eos if isinstance(eos, list) else [eos]

def complete(prompt):
    # Greedy continuation of the prompt: (text, number of new tokens, stopped by itself?)
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    new = out[0, inputs["input_ids"].shape[1]:]                 # only the generated tokens
    stopped = new[-1].item() in end_of_text_ids()
    return tokenizer.decode(new, skip_special_tokens=True), len(new), stopped

The questions asked to every model. `repr` shows line breaks as `\n`.
Add your own questions to the list.

In [2]:
QUESTIONS = [
    "What is the capital of France?",
    "Give me three tips to stay healthy.",
    "Write a short poem about the sea.",
    "What are the symptoms of the flu?",
    "Translate to French: Good morning, my friend.",
]

ANSWERS = {}     # model / prompt format → {question: (answer, stopped by itself?)}, for section 5

def ask_all(make_prompt, label):
    # Asks every question, with the prompt built by make_prompt(question).
    ANSWERS[label] = {}
    for question in QUESTIONS:
        text, n_tokens, stopped = complete(make_prompt(question))
        ANSWERS[label][question] = (text, stopped)
        end = "stopped by itself" if stopped else "cut at the length limit"
        print(f"▶ {question}   [{n_tokens} tokens, {end}]")
        print(textwrap.indent(textwrap.fill(repr(text), 100), "    "), "\n")

def bar(p, width=40):
    filled = round(p * width)
    return "█" * filled + "░" * (width - filled)

def next_tokens(prompt, k=10):
    # The k most probable NEXT tokens after the prompt (softmax of the last logits).
    ids = tokenizer(prompt, return_tensors="pt")["input_ids"]
    with torch.no_grad():
        probs = torch.softmax(model(ids).logits[0, -1], dim=-1)
    top = torch.topk(probs, k)
    return pd.DataFrame({"token": [repr(tokenizer.decode(i)) for i in top.indices],
                         "probability": top.values.double().numpy().round(3),
                         "": [bar(p.item()) for p in top.values]},
                        index=range(1, k + 1))

## 2. A base model: it continues the text

**Choose the base model.** `distilgpt2` is the model of the previous
notebook; `gpt2` is the exact model that `LaMini-GPT-124M` (part 3) started
from. To switch, swap the comment and run the notebook again.

In [3]:
BASE_MODEL = "distilgpt2"     # 82M, 6 layers (previous notebook)
# BASE_MODEL = "gpt2"         # 124M, 12 layers: LaMini-GPT-124M before its fine-tuning

load(BASE_MODEL)

distilgpt2: 82M parameters, 6 layers, vocabulary of 50,257 tokens


### 2.1 The question alone

The prompt is just the question.

In [4]:
def as_question(question):
    return question

ask_all(as_question, f"{BASE_MODEL}: question alone")

▶ What is the capital of France?   [60 tokens, cut at the length limit]
    '\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\
    n\n\n\n\n\n\n\n\n\n\n' 



▶ Give me three tips to stay healthy.   [60 tokens, cut at the length limit]
    '\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\
    n\n\n\n\n\n\n\n\n\n\n' 



▶ Write a short poem about the sea.   [60 tokens, cut at the length limit]
    '\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\
    n\n\n\n\n\n\n\n\n\n\n' 



▶ What are the symptoms of the flu?   [60 tokens, cut at the length limit]
    '\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\
    n\n\n\n\n\n\n\n\n\n\n' 



▶ Translate to French: Good morning, my friend.   [60 tokens, cut at the length limit]
    '\n\nI’m sorry, but I’m sorry, but I’m sorry, but I’m sorry, but I’m sorry, but I’m sorry, but I’m
    sorry, but I’m sorry, but I�' 



### 2.2 Why: what comes next after a question?

The next-token probabilities, as in the previous notebook. On the web, what
usually follows a question? For the model, the question is the start of a
document — a title, a forum post, a list of questions — not a request.

In [5]:
next_tokens(QUESTIONS[0])

,token,probability,
1,'\n',0.237,█████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
2,'�',0.066,███░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
3,' It',0.043,██░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
4,' The',0.041,██░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
5,' I',0.025,█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
6,' And',0.025,█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
7,' What',0.023,█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
8,' Is',0.023,█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
9,' A',0.015,█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
10,' In',0.014,█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░


### 2.3 Disguising the question as a document

Before instruction tuning, the usual trick was to write the beginning of a
document in which an answer is the natural continuation: here, a
question-and-answer page. This is **prompt engineering**: the model is the
same, only the text changes.

In [6]:
def as_qa(question):
    return f"Question: {question}\nAnswer:"

print(repr(as_qa(QUESTIONS[0])), "\n")
ask_all(as_qa, f"{BASE_MODEL}: Question/Answer")

'Question: What is the capital of France?\nAnswer:' 



▶ What is the capital of France?   [60 tokens, cut at the length limit]
    ' The capital of France is the capital of France.\nThe capital of France is the capital of
    France.\nThe capital of France is the capital of France.\nThe capital of France is the capital of
    France.\nThe capital of France is the capital of France.\nThe capital of France is' 



▶ Give me three tips to stay healthy.   [60 tokens, cut at the length limit]
    ' If you’re a healthy person, you’re not going to be able to eat.\nAnswer: If you’re a healthy
    person, you’re not going to be able to eat.\nAnswer: If you’re a healthy person, you’' 



▶ Write a short poem about the sea.   [60 tokens, cut at the length limit]
    ' I am a writer. I am a writer. I am a writer. I am a writer. I am a writer. I am a writer. I am a
    writer. I am a writer. I am a writer. I am a writer. I am a writer. I am a writer.' 



▶ What are the symptoms of the flu?   [60 tokens, cut at the length limit]
    ' The flu is a very common form of flu. It is a very common form of flu. It is a very common form of
    flu. It is a very common form of flu. It is a very common form of flu. It is a very common form of
    flu. It is a very common' 



▶ Translate to French: Good morning, my friend.   [60 tokens, cut at the length limit]
    ' I am a French citizen. I am a French citizen. I am a French citizen. I am a French citizen. I am a
    French citizen. I am a French citizen. I am a French citizen. I am a French citizen. I am a French
    citizen. I am a French citizen.' 



### 2.4 The template of an instruction-tuned model

`LaMini-GPT-124M` (part 3) was fine-tuned with every example written in the
template below (the *Alpaca* template). What does the **base** model do with
the same template?

In [7]:
ALPACA_TEMPLATE = ("Below is an instruction that describes a task. "
                   "Write a response that appropriately completes the request.\n\n"
                   "### Instruction:\n{instruction}\n\n### Response:")

def as_alpaca(question):
    return ALPACA_TEMPLATE.format(instruction=question)

print(as_alpaca(QUESTIONS[0]), "\n")
ask_all(as_alpaca, f"{BASE_MODEL}: Alpaca template")

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is the capital of France?

### Response: 



▶ What is the capital of France?   [60 tokens, cut at the length limit]
    '\nWhat is the capital of France?\n### Response:\nWhat is the capital of France?\n###
    Response:\nWhat is the capital of France?\n### Response:\nWhat is the capital of France?\n###
    Response:\nWhat is the capital of France?\n### Response:' 



▶ Give me three tips to stay healthy.   [60 tokens, cut at the length limit]
    '\nWrite a response that is not too long.\n### Response:\nWrite a response that is not too
    long.\n### Response:\nWrite a response that is not too long.\n### Response:\nWrite a response that
    is not too long.\n### Response:\nWrite a response' 



▶ Write a short poem about the sea.   [60 tokens, cut at the length limit]
    '\nWrite a short poem about the sea.\n### Response:\nWrite a short poem about the sea.\n###
    Response:\nWrite a short poem about the sea.\n### Response:\nWrite a short poem about the sea.\n###
    Response:\nWrite a short poem about the sea' 



▶ What are the symptoms of the flu?   [60 tokens, cut at the length limit]
    '\nWhat is the cause of the flu?\n### Response:\nWhat is the cause of the flu?\n### Response:\nWhat
    is the cause of the flu?\n### Response:\nWhat is the cause of the flu?\n### Response:\nWhat is the
    cause of the flu' 



▶ Translate to French: Good morning, my friend.   [60 tokens, cut at the length limit]
    '\nHello,\nHello,\nHello,\nHello,\nHello,\nHello,\nHello,\nHello,\nHello,\nHello,\nHello,\nHello,\nH
    ello,\nHello,\nHello,\nHello,\nHello,\nHello,\nHello,\nHello,' 



## 3. The same model after instruction fine-tuning

### 3.1 What the fine-tuning trains

Each training example is an instruction and a response written in the
template. The loss is the **same** next-token loss of pre-training
(cross-entropy), with one change: the tokens of the prompt get the label
`-100`, which PyTorch ignores, so the model learns only to **produce the
response** — and the end-of-text token after it, so it also learns to stop.

The example below was written for this notebook. The cell computes that
loss for the base model and for the fine-tuned one. A lower loss means the
model gives a higher probability to each token of this good response;
$e^{-\text{loss}}$ is the average (geometric mean) probability per token.

In [8]:
INSTRUCT_MODEL = "MBZUAI/LaMini-GPT-124M"

EXAMPLE_INSTRUCTION = "Name the three primary colors."
EXAMPLE_RESPONSE = "The three primary colors are red, yellow and blue."

def response_loss(instruction, response):
    prompt_ids = tokenizer(as_alpaca(instruction), return_tensors="pt")["input_ids"]
    response_ids = tokenizer(response + tokenizer.eos_token, return_tensors="pt")["input_ids"]
    ids = torch.cat([prompt_ids, response_ids], dim=1)       # prompt + response + <|endoftext|>
    labels = ids.clone()
    labels[0, :prompt_ids.shape[1]] = -100                   # the prompt is not learned
    with torch.no_grad():
        loss = model(input_ids=ids, labels=labels).loss      # mean cross-entropy on the response
    return loss.item(), response_ids.shape[1]

print(as_alpaca(EXAMPLE_INSTRUCTION) + EXAMPLE_RESPONSE + tokenizer.eos_token, "\n")

rows = []
for name in [BASE_MODEL, INSTRUCT_MODEL]:
    load(name)
    loss, n = response_loss(EXAMPLE_INSTRUCTION, EXAMPLE_RESPONSE)
    rows.append({"model": name, "response tokens scored": n, "loss": round(loss, 2),
                 "average probability per token": round(torch.exp(torch.tensor(-loss)).item(), 3)})
pd.DataFrame(rows).set_index("model")

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Name the three primary colors.

### Response:The three primary colors are red, yellow and blue.<|endoftext|> 



distilgpt2: 82M parameters, 6 layers, vocabulary of 50,257 tokens


MBZUAI/LaMini-GPT-124M: 124M parameters, 12 layers, vocabulary of 50,258 tokens


,response tokens scored,loss,average probability per token
model,,,
distilgpt2,12,2.68,0.069
MBZUAI/LaMini-GPT-124M,12,0.75,0.473


Training (not done here) would compute this loss over millions of
examples, and change the weights by gradient descent to make it smaller:
`loss.backward()` and an optimizer step, as in pre-training. The
`INSTRUCT_MODEL` is now loaded.

### 3.2 Answers with the template

In [9]:
ask_all(as_alpaca, f"{INSTRUCT_MODEL}: Alpaca template")

▶ What is the capital of France?   [3 tokens, stopped by itself]
    'Paris.' 



▶ Give me three tips to stay healthy.   [29 tokens, stopped by itself]
    '1. Eat a balanced and nutritious diet.\n2. Drink plenty of water and electrolytes.\n3. Practice
    good sleep habits.' 



▶ Write a short poem about the sea.   [60 tokens, cut at the length limit]
    'The sea is vast and vast,\nIts depths vast and vast.\nIts waves crash and crash,\nIts waves crash
    and crash,\nIts waves crash and crash,\nIts waves crash and crash,\nIts waves crash and
    crash,\n\nThe sea is a wondrous place,\n' 



▶ What are the symptoms of the flu?   [26 tokens, stopped by itself]
    'Fever, cough, sore throat, body aches, fatigue, and body aches are common symptoms of the flu.' 



▶ Translate to French: Good morning, my friend.   [10 tokens, stopped by itself]
    "Je suis de l'aide." 



### 3.3 Next tokens after the template

The same question as in 2.2, now inside the template.

In [10]:
next_tokens(as_alpaca(QUESTIONS[0]))

,token,probability,
1,'Paris',0.497,████████████████████░░░░░░░░░░░░░░░░░░░░
2,'The',0.141,██████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
3,'Saint',0.074,███░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
4,'Be',0.027,█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
5,'R',0.024,█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
6,'B',0.021,█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
7,'Mad',0.020,█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
8,'Pr',0.018,█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
9,'Mont',0.018,█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
10,'Le',0.015,█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░


### 3.4 The fine-tuned model without its template

The model learned to answer **after `### Response:`**. Without the
template, the question alone:

In [11]:
ask_all(as_question, f"{INSTRUCT_MODEL}: question alone")

▶ What is the capital of France?   [1 tokens, stopped by itself]
    '' 



▶ Give me three tips to stay healthy.   [40 tokens, stopped by itself]
    '\n\n1. Eat a balanced diet with plenty of fruits and vegetables.\n2. Avoid processed and high-sugar
    foods.\n3. Avoid sugary drinks and excessive alcohol consumption.' 

▶ Write a short poem about the sea.   [1 tokens, stopped by itself]
    '' 

▶ What are the symptoms of the flu?   [1 tokens, stopped by itself]
    '' 



▶ Translate to French: Good morning, my friend.   [26 tokens, stopped by itself]
    " I'm sorry, I'm not sure what you mean by that. Can you please provide more context or clarify your
    question?" 



## 4. A chat model

A **chat** model is an instruction-tuned model whose template marks a
conversation: who is speaking (`system`, `user`, `assistant`) and where
each turn ends. SmolLM2 (Hugging Face, 2024) has a base and an instruct
version of 135M parameters, trained on far more text than GPT-2 (2 trillion
tokens).

### 4.1 The base model

In [12]:
CHAT_BASE_MODEL = "HuggingFaceTB/SmolLM2-135M"
CHAT_MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"

load(CHAT_BASE_MODEL)
ask_all(as_question, f"{CHAT_BASE_MODEL}: question alone")

HuggingFaceTB/SmolLM2-135M: 135M parameters, 30 layers, vocabulary of 49,152 tokens


▶ What is the capital of France?   [60 tokens, cut at the length limit]
    '\n\nThe capital of France is Paris.\n\nWhat is the capital of France?\n\nParis is the capital of
    France.\n\nWhat is the capital of France?\n\nParis is the capital of France.\n\nWhat is the capital
    of France?\n\nParis is the capital' 



▶ Give me three tips to stay healthy.   [60 tokens, cut at the length limit]
    '\n\nThe first tip is to eat a healthy diet.\n\nThe second tip is to exercise regularly.\n\nThe
    third tip is to get enough sleep.\n\nThe first tip is to eat a healthy diet.\n\nThe second is to
    exercise regularly.\n\nThe third is to' 



▶ Write a short poem about the sea.   [60 tokens, cut at the length limit]
    '\n\nThe sea is a very important part of our lives. It is the place where we go to school, to work,
    to play, to relax, to relax. It is the place where we go to sleep, to wake up, to go to school, to
    work, to play,' 



▶ What are the symptoms of the flu?   [60 tokens, cut at the length limit]
    '\n\nThe flu is a respiratory illness caused by a virus. It is a common illness that affects people
    of all ages. The flu is most common in the winter and spring. It is caused by a virus that is spread
    by coughing, sneezing, and contact with infected people.\n\nThe flu' 



▶ Translate to French: Good morning, my friend.   [60 tokens, cut at the length limit]
    '\n\nThe word "good" is a verb, and the verb "translate" is a noun.\n\nThe verb "translate" is a
    noun, and the noun "good" is a verb.\n\nThe verb "translate" is a noun, and the noun "good" is' 



### 4.2 The chat template

The conversation is a list of messages. The tokenizer of the instruct
model turns it into **one text** with its template
(`apply_chat_template`): special tokens `<|im_start|>` and `<|im_end|>`
mark each message, a default `system` message is added, and the text ends
with the start of the `assistant` turn — the model continues from there.

In [13]:
load(CHAT_MODEL)

def chat_prompt(messages):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def as_chat(question):
    return chat_prompt([{"role": "user", "content": question}])

print(as_chat(QUESTIONS[0]))

ids = tokenizer(as_chat(QUESTIONS[0]))["input_ids"]
special = {t: tokenizer.convert_tokens_to_ids(t) for t in ["<|im_start|>", "<|im_end|>"]}
print(f"{len(ids)} tokens; special tokens (one id each): {special}; "
      f"end-of-text for generation: {end_of_text_ids()}")

HuggingFaceTB/SmolLM2-135M-Instruct: 135M parameters, 30 layers, vocabulary of 49,152 tokens
<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant

37 tokens; special tokens (one id each): {'<|im_start|>': 1, '<|im_end|>': 2}; end-of-text for generation: [2]


### 4.3 Answers

In [14]:
ask_all(as_chat, f"{CHAT_MODEL}: chat template")

▶ What is the capital of France?   [8 tokens, stopped by itself]
    'The capital of France is Paris.' 



▶ Give me three tips to stay healthy.   [60 tokens, cut at the length limit]
    '1. Eat a balanced diet: A healthy diet is essential for maintaining good health. Include a variety
    of fruits, vegetables, lean proteins, and whole grains in your meals.\n\n2. Stay hydrated: Drink
    plenty of water throughout the day. Dehydration can lead to a range of health issues.' 



▶ Write a short poem about the sea.   [60 tokens, cut at the length limit]
    'The waves crash against the shore,\nA symphony of sound, a symphony of life.\nThe salty spray
    dances, a dance of light,\nA dance of the sea, a dance of life.\n\nThe sun dips below the
    horizon,\nA gentle caress, a gentle caress' 



▶ What are the symptoms of the flu?   [60 tokens, cut at the length limit]
    'The symptoms of the flu can vary from person to person, but common signs include:\n\n1. Fever: A
    high body temperature (usually above 100.4°F or 38°C)\n2. Cough: A dry, hacking cough that produces
    mucus\n3.' 



▶ Translate to French: Good morning, my friend.   [8 tokens, stopped by itself]
    "Je m'appelle Pierre." 



### 4.4 A conversation

The model has **no memory** between calls. A conversation is the whole
history sent again each time: each answer is appended as an `assistant`
message, then the next `user` message.

In [15]:
conversation = []
for user_turn in ["What is the capital of France?", "And of Italy?"]:
    conversation.append({"role": "user", "content": user_turn})
    answer, _, _ = complete(chat_prompt(conversation))
    conversation.append({"role": "assistant", "content": answer})

print(chat_prompt(conversation[:-1]))     # what the model received for the last answer
print(answer)

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant
The capital of France is Paris.<|im_end|>
<|im_start|>user
And of Italy?<|im_end|>
<|im_start|>assistant

The capital of Italy is Rome.


## 5. All the answers

**Did the model stop by itself?** ✔ = it chose its end-of-text token;
✘ = it was cut at `MAX_NEW_TOKENS`. One row per model and prompt format.

In [16]:
pd.DataFrame({label: {q: "✔" if stopped else "✘" for q, (_, stopped) in answers.items()}
              for label, answers in ANSWERS.items()}).T

,What is the capital of France?,Give me three tips to stay healthy.,Write a short poem about the sea.,What are the symptoms of the flu?,"Translate to French: Good morning, my friend."
distilgpt2: question alone,✘,✘,✘,✘,✘
distilgpt2: Question/Answer,✘,✘,✘,✘,✘
distilgpt2: Alpaca template,✘,✘,✘,✘,✘
MBZUAI/LaMini-GPT-124M: Alpaca template,✔,✔,✘,✔,✔
MBZUAI/LaMini-GPT-124M: question alone,✔,✔,✔,✔,✔
HuggingFaceTB/SmolLM2-135M: question alone,✘,✘,✘,✘,✘
HuggingFaceTB/SmolLM2-135M-Instruct: chat template,✔,✘,✘,✘,✔


**One question, all the models**: the beginning of each answer (80
characters). Change `SUMMARY_QUESTION` to see another one.

In [17]:
SUMMARY_QUESTION = QUESTIONS[0]

pd.DataFrame({"answer": {label: repr(answers[SUMMARY_QUESTION][0])[:80]
                         for label, answers in ANSWERS.items()}})

,answer
distilgpt2: question alone,'\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\
distilgpt2: Question/Answer,' The capital of France is the capital of France.\nThe capital of France is the
distilgpt2: Alpaca template,'\nWhat is the capital of France?\n### Response:\nWhat is the capital of France?
MBZUAI/LaMini-GPT-124M: Alpaca template,'Paris.'
MBZUAI/LaMini-GPT-124M: question alone,''
HuggingFaceTB/SmolLM2-135M: question alone,'\n\nThe capital of France is Paris.\n\nWhat is the capital of France?\n\nParis
HuggingFaceTB/SmolLM2-135M-Instruct: chat template,'The capital of France is Paris.'


## What to notice (base model `distilgpt2`)

* **The base model continues, it does not answer.** With the question
  alone, `distilgpt2` writes 60 line breaks for four of the five questions.
  The most probable token after *What is the capital of France?* is a line
  break (0.237): on the web, a question is often a title, followed by a new
  line. (The second candidate, `'�'`, is half of a character written in
  two tokens, such as a curly quote.)
* **Prompt engineering changes the form, not the behaviour.** In the
  *Question/Answer* format the model writes answer-shaped sentences, but
  empty or wrong (*The capital of France is the capital of France.*), and
  repeats them. With the Alpaca template it copies the template itself
  (`### Response:` again and again). None of the 15 base answers ends by
  itself: all are cut at the length limit.
* **Fine-tuning: the same loss, other data.** On the example response of
  3.1, the loss goes from 2.68 (`distilgpt2`) to 0.75 (`LaMini-GPT-124M`):
  the average probability of each token of the response, from 0.069 to
  0.473.
* **The fine-tuned model answers, and stops.** *Paris.* in 3 tokens; four
  of the five answers end with the end-of-text token. Right after
  `### Response:`, the most probable token is already *Paris* (0.497). The
  poem still falls into a loop (*vast and vast*, *crash and crash*).
* **The template is part of what was learned.** Without it, the same
  model chooses end-of-text at once for three questions (empty answers),
  still lists the three tips, and answers the translation with *I'm sorry,
  I'm not sure what you mean*.
* **A chat template is an instruction template with roles.** The
  seven-word question becomes 37 tokens: the default `system` message,
  `<|im_start|>` (id 1) and `<|im_end|>` (id 2) around each message, and
  the opening of the `assistant` turn. `<|im_end|>` is also the token that
  ends generation.
* **A better base model knows more, but still does not answer.** The base
  SmolLM2-135M, trained on 2 trillion tokens, writes *The capital of France
  is Paris.* — and then the question again, as in a FAQ page; all five
  answers are cut at the limit. The instruct version answers in 8 tokens
  and stops; its longer answers (tips, poem, flu) are cut only because 60
  tokens are not enough for a list.
* **A conversation is one long prompt.** *And of Italy?* is answered (*The
  capital of Italy is Rome.*) only because the whole history is sent again;
  the model itself keeps nothing between calls.
* **Following instructions is not knowing.** Both instruct models
  "translate" confidently and wrongly: *Je suis de l'aide.* (*I am help*)
  and *Je m'appelle Pierre.* (*My name is Pierre*). The fine-tuning teaches
  the form and the behaviour of a response; what the model knows comes
  from pre-training, and 124–135M parameters know little.

## Questions

1. In 2.2, which token is the most probable after the question? What kind
   of web page would explain it?
2. The fine-tuned `LaMini-GPT-124M` and the base `gpt2` have the same
   architecture and the same number of parameters. Switch `BASE_MODEL` to
   `gpt2` and run again: what is different between them, and what is the
   same?
3. In 3.1, which tokens are scored by the loss and which are not? Why must
   the end-of-text token be part of the response?
4. Change the conversation of 4.4: add a `system` message at the start
   (e.g. *Answer with one word only*). Does a 135M model follow it?
5. Some answers are well formed but wrong. Find one. Did the fine-tuning
   teach the model new facts, or a new behaviour?